In [1]:
%pip install -q "torch>=2.2" "scikit-learn>=1.2" "imbalanced-learn>=0.11" pandas numpy joblib matplotlib

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os, json, random, warnings, copy
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, joblib
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, average_precision_score, confusion_matrix, precision_recall_curve
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from IPython.display import display
warnings.filterwarnings("ignore", category=UserWarning)
SEED=1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_PATH=Path("secom_telemetry_modified.csv")
if not DATA_PATH.exists():
    try:
        from google.colab import files
        print("Upload secom_telemetry_modified.csv"); files.upload()
    except ImportError: pass
assert DATA_PATH.exists(), f"Missing {DATA_PATH.resolve()}"
print("torch",torch.__version__,"device",DEVICE)

torch 2.11.0+cpu device cpu


In [4]:
df0=pd.read_csv(DATA_PATH)
SENSOR_COLS=[c for c in df0.columns if c.startswith("sensor_")]
assert "timestamp" in df0 and "label" in df0
X_all=df0[SENSOR_COLS].apply(pd.to_numeric,errors="coerce").to_numpy(float)
y_all=(df0.label.to_numpy()==1).astype(int)
print("shape",X_all.shape,"positives",y_all.sum(),"natural prevalence",y_all.mean())
imp_bad=SimpleImputer(strategy="mean",keep_empty_features=True); Xi=imp_bad.fit_transform(X_all)
sc_bad=StandardScaler(); Xs=sc_bad.fit_transform(Xi)
sm_bad=SMOTE(random_state=SEED,k_neighbors=5); Xb,yb=sm_bad.fit_resample(Xs,y_all)
X_train_bad,X_test_bad,y_train_bad,y_test_bad=train_test_split(Xb,yb,test_size=.2,stratify=yb,random_state=SEED)
print("SMOTE-before-split invalid test prevalence:",y_test_bad.mean(),"test n",len(y_test_bad))

shape (1567, 590) positives 104 natural prevalence 0.06636885768985322
SMOTE-before-split invalid test prevalence: 0.5 test n 586


In [5]:
df=pd.read_csv(DATA_PATH)
SENSOR_COLS=[c for c in df.columns if c.startswith("sensor_")]
df["timestamp"]=pd.to_datetime(df.timestamp,errors="coerce")
assert df.timestamp.notna().all()
df=df.sort_values("timestamp",kind="stable").reset_index(drop=True)
numeric=df[SENSOR_COLS].apply(pd.to_numeric,errors="coerce")
y=(df.label.to_numpy()==1).astype(int)
legacy=df.sensor_0.eq(0.0)
print("shape",df.shape,"time range",df.timestamp.min(),"to",df.timestamp.max())
print("label counts",pd.Series(y).value_counts().to_dict(),"defect rate",y.mean())
print("missing cell rate",numeric.isna().mean().mean())
print("legacy count",legacy.sum())
display(df.loc[legacy,"sensor_1"].value_counts(dropna=False).sort_index())
display(numeric.loc[~legacy,"sensor_1"].describe())
constant_all=numeric.columns[numeric.std().fillna(0).eq(0)].tolist()
print("constant sensors",len(constant_all),constant_all[:30])
display(numeric.isna().mean().sort_values(ascending=False).head(20).to_frame("missing_rate"))

shape (1567, 592) time range 2008-07-19 11:55:00 to 2008-10-17 06:07:00
label counts {0: 1463, 1: 104} defect rate 0.06636885768985322
missing cell rate 0.04537224319383903
legacy count 146


,count
sensor_1,
100.0,61
200.0,35
300.0,50


,sensor_1
count,1417.000000
mean,2495.685956
std,79.820943
min,2158.750000
25%,2452.280000
50%,2499.150000
75%,2539.500000
max,2846.440000


constant sensors 116 ['sensor_5', 'sensor_13', 'sensor_42', 'sensor_49', 'sensor_52', 'sensor_69', 'sensor_97', 'sensor_141', 'sensor_149', 'sensor_178', 'sensor_179', 'sensor_186', 'sensor_189', 'sensor_190', 'sensor_191', 'sensor_192', 'sensor_193', 'sensor_194', 'sensor_226', 'sensor_229', 'sensor_230', 'sensor_231', 'sensor_232', 'sensor_233', 'sensor_234', 'sensor_235', 'sensor_236', 'sensor_237', 'sensor_240', 'sensor_241']


,missing_rate
sensor_292,0.911934
sensor_293,0.911934
sensor_158,0.911934
sensor_157,0.911934
sensor_492,0.855775
sensor_85,0.855775
sensor_358,0.855775
sensor_220,0.855775
sensor_244,0.649649
sensor_517,0.649649


In [6]:
bad_clf=LogisticRegression(max_iter=1000,random_state=SEED).fit(X_train_bad,y_train_bad)
pb=bad_clf.predict_proba(X_test_bad)[:,1]; pred_b=(pb>=.5).astype(int)
bad_result={"split":"SMOTE before split (INVALID)","prevalence":float(y_test_bad.mean()),"F1":f1_score(y_test_bad,pred_b,zero_division=0),"precision":precision_score(y_test_bad,pred_b,zero_division=0),"recall":recall_score(y_test_bad,pred_b,zero_division=0)}
Xtr0,Xte0,ytr0,yte0=train_test_split(X_all,y_all,test_size=.2,stratify=y_all,random_state=SEED)
imp=SimpleImputer(strategy="median",keep_empty_features=True); Xtr=imp.fit_transform(Xtr0); Xte=imp.transform(Xte0)
sc=StandardScaler(); Xtr=sc.fit_transform(Xtr); Xte=sc.transform(Xte)
Xtrs,ytrs=SMOTE(random_state=SEED,k_neighbors=5).fit_resample(Xtr,ytr0)
good_clf=LogisticRegression(max_iter=1000,random_state=SEED).fit(Xtrs,ytrs)
pg=good_clf.predict_proba(Xte)[:,1]; pred_g=(pg>=.5).astype(int)
good_result={"split":"SMOTE train only, natural holdout","prevalence":float(yte0.mean()),"F1":f1_score(yte0,pred_g,zero_division=0),"precision":precision_score(yte0,pred_g,zero_division=0),"recall":recall_score(yte0,pred_g,zero_division=0)}
display(pd.DataFrame([bad_result,good_result]))
print("Caution: these are diagnostic comparisons, not an apples-to-apples score comparison: the invalid row is evaluated on a balanced, synthetic-contaminated test split; the train-only row is evaluated on a natural-prevalence random holdout.")


,split,prevalence,F1,precision,recall
0,SMOTE before split (INVALID),0.500000,0.938907,0.887538,0.996587
1,"SMOTE train only, natural holdout",0.066879,0.076923,0.064516,0.095238


Caution: these are diagnostic comparisons, not an apples-to-apples score comparison: the invalid row is evaluated on a balanced, synthetic-contaminated test split; the train-only row is evaluated on a natural-prevalence random holdout.


In [7]:
cut=int(.60*len(df)); Xnum=df[SENSOR_COLS].apply(pd.to_numeric,errors="coerce")
it=SimpleImputer(strategy="median",keep_empty_features=True); Xt=it.fit_transform(Xnum.iloc[:cut])
st=StandardScaler().fit(Xt)
ifull=SimpleImputer(strategy="mean",keep_empty_features=True); Xf=ifull.fit_transform(Xnum)
sf=StandardScaler().fit(Xf)
statdiff=pd.DataFrame({"train_imputer":it.statistics_,"full_mean_imputer":ifull.statistics_,
                       "imputer_abs_diff":np.abs(it.statistics_-ifull.statistics_),
                       "train_scaler_mean":st.mean_,"full_scaler_mean":sf.mean_,
                       "scaler_mean_abs_diff":np.abs(st.mean_-sf.mean_),
                       "train_scale":st.scale_,"full_scale":sf.scale_,
                       "scale_abs_diff":np.abs(st.scale_-sf.scale_)},index=SENSOR_COLS)
display(statdiff[["imputer_abs_diff","scaler_mean_abs_diff","scale_abs_diff"]].describe())
display(statdiff.sort_values("scaler_mean_abs_diff",ascending=False).head(10))

,imputer_abs_diff,scaler_mean_abs_diff,scale_abs_diff
count,590.000000,590.000000,590.000000
mean,19.547325,4.537331,5.261804
std,181.282518,36.728050,28.276874
min,0.000000,0.000000,0.000000
25%,0.000289,0.000109,0.000234
50%,0.017054,0.011302,0.015899
75%,0.710737,0.306766,0.842533
max,3669.154633,711.732292,504.221438


,train_imputer,full_mean_imputer,imputer_abs_diff,train_scaler_mean,full_scaler_mean,scaler_mean_abs_diff,train_scale,full_scale,scale_abs_diff
sensor_162,1128.00000,4797.154633,3669.154633,4085.422340,4797.154633,711.732292,6397.263860,6547.292953,150.029092
sensor_297,517.67395,2342.826978,1825.153028,1983.739404,2342.826978,359.087575,3133.211980,3223.833867,90.621887
sensor_24,255.12500,-298.598136,553.723136,26.327483,-298.598136,324.925619,2807.446636,2899.910206,92.463570
sensor_161,2825.50000,4066.850479,1241.350479,4190.363830,4066.850479,123.513351,4354.186567,4235.185125,119.001441
sensor_160,508.50000,555.346326,46.846326,664.509574,555.346326,109.163249,689.269104,574.258093,115.011011
sensor_159,580.50000,882.680511,302.180511,963.080851,882.680511,80.400340,1204.225145,982.101559,222.123585
sensor_21,-5595.00000,-5618.393610,23.393610,-5686.888032,-5618.393610,68.494422,750.313642,626.221870,124.091772
sensor_3,1275.09170,1396.376627,121.284927,1334.658663,1396.376627,61.717964,327.858595,439.572525,111.713931
sensor_296,1253.20650,1879.228369,626.021869,1939.157300,1879.228369,59.928931,2020.740714,1973.219798,47.520916
sensor_22,2727.50000,2699.378435,28.121565,2758.752128,2699.378435,59.373693,350.353938,295.215535,55.138403


In [8]:
def eval_logistic(Xtr0,ytr,Xte0,yte,name):
    imp=SimpleImputer(strategy="median",keep_empty_features=True); a=imp.fit_transform(Xtr0); b=imp.transform(Xte0)
    sc=StandardScaler(); a=sc.fit_transform(a); b=sc.transform(b)
    model=LogisticRegression(max_iter=1000,class_weight="balanced",random_state=SEED).fit(a,ytr)
    p=model.predict_proba(b)[:,1]; q=(p>=.5).astype(int)
    return {"split":name,"n_test":len(yte),"prevalence":yte.mean(),"F1@0.5":f1_score(yte,q,zero_division=0),
            "precision":precision_score(yte,q,zero_division=0),"recall":recall_score(yte,q,zero_division=0),
            "PR-AUC":average_precision_score(yte,p)}
Xmat=df[SENSOR_COLS].apply(pd.to_numeric,errors="coerce").to_numpy(float)
Xa,Xb,ya,yb=train_test_split(Xmat,y,test_size=.2,stratify=y,random_state=SEED)
random_result=eval_logistic(Xa,ya,Xb,yb,"random stratified")
c=int(.8*len(df)); time_result=eval_logistic(Xmat[:c],y[:c],Xmat[c:],y[c:],"chronological last 20%")
display(pd.DataFrame([random_result,time_result]))
print("Interpret differences as diagnostic only; prevalence and temporal drift may differ.")

,split,n_test,prevalence,F1@0.5,precision,recall,PR-AUC
0,random stratified,314,0.066879,0.111111,0.090909,0.142857,0.085525
1,chronological last 20%,314,0.054140,0.176471,0.176471,0.176471,0.179263


Interpret differences as diagnostic only; prevalence and temporal drift may differ.


In [9]:
B,F_demo=64,len(SENSOR_COLS)
reconstructed=torch.zeros(B,F_demo); batch_x=torch.zeros(B,1,F_demo)
print("reconstructed",tuple(reconstructed.shape))
print("batch_x",tuple(batch_x.shape))
print("difference",tuple((reconstructed-batch_x).shape))
assert tuple((reconstructed-batch_x).shape)==(64,64,F_demo)

reconstructed (64, 590)
batch_x (64, 1, 590)
difference (64, 64, 590)


In [10]:
s1=pd.to_numeric(df.sensor_1,errors="coerce")
print("legacy sensor_1 counts"); display(s1[legacy].value_counts(dropna=False).sort_index())
print("sensor_1 overall raw mean/std",s1.mean(),s1.std())
print("sensor_1 legacy raw mean/std",s1[legacy].mean(),s1[legacy].std())
demo=pd.DataFrame({"sensor_0":pd.to_numeric(df.sensor_0,errors="coerce"),"sensor_1":s1})
demo_imp=SimpleImputer(strategy="mean",keep_empty_features=True).fit_transform(demo)
demo_sc=StandardScaler().fit_transform(demo_imp)
print("naive imputed means",demo_imp.mean(axis=0),"stds",demo_imp.std(axis=0))
print("Naive scaling treats sentinel and 100/200/300 codes as continuous values.")

legacy sensor_1 counts


,count
sensor_1,
100.0,61
200.0,35
300.0,50


sensor_1 overall raw mean/std 2280.5419065898914 675.2821153971395
sensor_1 legacy raw mean/std 192.46575342465752 87.16659945450964
naive imputed means [2732.11646381 2280.54190659] stds [878.76215714 674.203906  ]
Naive scaling treats sentinel and 100/200/300 codes as continuous values.


In [11]:
n=len(df); train_end=int(.60*n); val_end=int(.80*n)
train_df=df.iloc[:train_end].copy(); val_df=df.iloc[train_end:val_end].copy(); test_df=df.iloc[val_end:].copy()
def get_y(frame): return (frame.label.to_numpy()==1).astype(np.int64)
y_train,y_val,y_test=map(get_y,[train_df,val_df,test_df])
print("sizes",len(train_df),len(val_df),len(test_df))
print("rates",{"train":y_train.mean(),"validation":y_val.mean(),"test":y_test.mean(),"overall":y.mean()})
print("time ranges",[(x.timestamp.min(),x.timestamp.max()) for x in [train_df,val_df,test_df]])
assert train_df.timestamp.max()<=val_df.timestamp.min()<=test_df.timestamp.min()

sizes 940 313 314
rates {'train': np.float64(0.08085106382978724), 'validation': np.float64(0.03514376996805112), 'test': np.float64(0.054140127388535034), 'overall': np.float64(0.06636885768985322)}
time ranges [(Timestamp('2008-07-19 11:55:00'), Timestamp('2008-09-20 05:34:00')), (Timestamp('2008-09-20 06:08:00'), Timestamp('2008-10-02 19:25:00')), (Timestamp('2008-10-02 20:54:00'), Timestamp('2008-10-17 06:07:00'))]


In [12]:
LEGACY_FLAG="is_legacy"; CODE_COLS=["code_100","code_200","code_300"]
def split_features(frame):
    out=frame[SENSOR_COLS].apply(pd.to_numeric,errors="coerce").copy()
    mask=frame.sensor_0.eq(0.0).fillna(False)
    s1=out.sensor_1
    flags=pd.DataFrame({"is_legacy":mask.astype("float32").to_numpy(),
      "code_100":(mask&s1.eq(100)).astype("float32").to_numpy(),
      "code_200":(mask&s1.eq(200)).astype("float32").to_numpy(),
      "code_300":(mask&s1.eq(300)).astype("float32").to_numpy()},index=frame.index)
    out.loc[mask,["sensor_0","sensor_1"]]=np.nan
    return out,flags
Xtr_raw,flag_tr=split_features(train_df); Xv_raw,flag_v=split_features(val_df); Xte_raw,flag_te=split_features(test_df)
imputer=SimpleImputer(strategy="median",keep_empty_features=True)
A=imputer.fit_transform(Xtr_raw); B=imputer.transform(Xv_raw); C=imputer.transform(Xte_raw)
std=np.nanstd(A,axis=0); keep=np.isfinite(std)&(std>0)
kept_cont_cols=[c for c,k in zip(SENSOR_COLS,keep) if k]
dropped_constant_cols=[c for c,k in zip(SENSOR_COLS,keep) if not k]
print("dropped train-constant columns",len(dropped_constant_cols),dropped_constant_cols[:40])
A=A[:,keep]; B=B[:,keep]; C=C[:,keep]
scaler=StandardScaler(); A=scaler.fit_transform(A).astype("float32")
B=scaler.transform(B).astype("float32"); C=scaler.transform(C).astype("float32")
def combine(x,flags): return np.concatenate([x,flags[["is_legacy"]+CODE_COLS].to_numpy(dtype="float32")],axis=1)
X_train,X_val,X_test=combine(A,flag_tr),combine(B,flag_v),combine(C,flag_te)
FEATURE_COLS=kept_cont_cols+["is_legacy"]+CODE_COLS
assert X_train.shape[1]==X_val.shape[1]==X_test.shape[1]==len(FEATURE_COLS)
assert np.isfinite(X_train).all() and np.isfinite(X_val).all() and np.isfinite(X_test).all()
print("feature shapes",X_train.shape,X_val.shape,X_test.shape)

dropped train-constant columns 122 ['sensor_5', 'sensor_13', 'sensor_42', 'sensor_49', 'sensor_52', 'sensor_69', 'sensor_74', 'sensor_97', 'sensor_141', 'sensor_149', 'sensor_178', 'sensor_179', 'sensor_186', 'sensor_189', 'sensor_190', 'sensor_191', 'sensor_192', 'sensor_193', 'sensor_194', 'sensor_206', 'sensor_209', 'sensor_226', 'sensor_229', 'sensor_230', 'sensor_231', 'sensor_232', 'sensor_233', 'sensor_234', 'sensor_235', 'sensor_236', 'sensor_237', 'sensor_240', 'sensor_241', 'sensor_242', 'sensor_243', 'sensor_256', 'sensor_257', 'sensor_258', 'sensor_259', 'sensor_260']
feature shapes (940, 472) (313, 472) (314, 472)


In [13]:
n_pos=int(y_train.sum()); n_neg=int((y_train==0).sum())
assert n_pos>0
POS_WEIGHT_VALUE=n_neg/n_pos
print("train positives",n_pos,"negatives",n_neg,"pos_weight",POS_WEIGHT_VALUE)

train positives 76 negatives 864 pos_weight 11.368421052631579


In [14]:
HP={"batch_size":64,"epochs":60,"lr":1e-3,"weight_decay":1e-5,"hidden":256,"latent":32,"lambda_clf":1.0,"patience":10}
def make_loader(X,y,shuffle=False):
    return DataLoader(TensorDataset(torch.tensor(X,dtype=torch.float32),torch.tensor(y,dtype=torch.float32)),
                      batch_size=HP["batch_size"],shuffle=shuffle,drop_last=False,
                      generator=torch.Generator().manual_seed(SEED) if shuffle else None)
train_loader=make_loader(X_train,y_train,True); val_loader=make_loader(X_val,y_val); test_loader=make_loader(X_test,y_test)
class WaferSAE(nn.Module):
    def __init__(self,d_in,d_hid=256,d_lat=32):
        super().__init__()
        self.encoder=nn.Sequential(nn.Linear(d_in,d_hid),nn.LayerNorm(d_hid),nn.GELU(),nn.Dropout(.1),
            nn.Linear(d_hid,d_hid//2),nn.LayerNorm(d_hid//2),nn.GELU(),nn.Linear(d_hid//2,d_lat))
        self.decoder=nn.Sequential(nn.Linear(d_lat,d_hid//2),nn.GELU(),nn.Linear(d_hid//2,d_hid),nn.GELU(),nn.Linear(d_hid,d_in))
        self.head=nn.Linear(d_lat,1)
    def forward(self,x):
        assert x.ndim==2, f"expected [B,F], got {tuple(x.shape)}"
        z=self.encoder(x); reconstruction=self.decoder(z); logits=self.head(z).squeeze(-1)
        assert reconstruction.shape==x.shape
        assert logits.shape==(x.shape[0],)
        return reconstruction,logits,z
model=WaferSAE(X_train.shape[1],HP["hidden"],HP["latent"]).to(DEVICE)
pos_weight=torch.tensor([POS_WEIGHT_VALUE],device=DEVICE,dtype=torch.float32)
bce=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer=torch.optim.AdamW(model.parameters(),lr=HP["lr"],weight_decay=HP["weight_decay"])
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=HP["epochs"])
def predict_probs(model,loader):
    model.eval(); probs=[]; ys=[]
    with torch.no_grad():
        for xb,yb in loader:
            _,logits,_=model(xb.to(DEVICE)); probs.extend(torch.sigmoid(logits).cpu().numpy()); ys.extend(yb.numpy())
    return np.asarray(probs),np.asarray(ys,dtype=int)
history={"train_total":[],"train_recon":[],"train_clf":[],"val_pr_auc":[]}
best_state=None; best_val_ap=-np.inf; best_epoch=0; patience=0
for epoch in range(1,HP["epochs"]+1):
    model.train(); sums=np.zeros(3); batches=0
    for xb,yb in train_loader:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE)
        rec,logits,_=model(xb)
        assert rec.shape==xb.shape and logits.shape==yb.shape
        loss_rec=((rec-xb)**2).mean(); loss_clf=bce(logits,yb)
        loss=loss_rec+HP["lambda_clf"]*loss_clf
        optimizer.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        sums += [loss.item(),loss_rec.item(),loss_clf.item()]; batches+=1
    scheduler.step()
    vp,vy=predict_probs(model,val_loader)
    ap=average_precision_score(vy,vp) if len(np.unique(vy))>1 else np.nan
    for key,val in zip(["train_total","train_recon","train_clf"],sums/batches): history[key].append(float(val))
    history["val_pr_auc"].append(float(ap))
    if np.isfinite(ap) and ap>best_val_ap:
        best_val_ap=float(ap); best_epoch=epoch; best_state=copy.deepcopy(model.state_dict()); patience=0
    else: patience+=1
    if epoch==1 or epoch%5==0: print(f"epoch {epoch}: total={sums[0]/batches:.4f}, recon={sums[1]/batches:.4f}, clf={sums[2]/batches:.4f}, val PR-AUC={ap:.4f}")
    if patience>=HP["patience"]: print("early stopping; best epoch",best_epoch); break
if best_state is None: raise RuntimeError("Could not select model: validation needs both classes.")
model.load_state_dict(best_state)
print("selected epoch",best_epoch,"validation PR-AUC",best_val_ap)

epoch 1: total=2.2318, recon=0.9888, clf=1.2429, val PR-AUC=0.0312
epoch 5: total=0.9491, recon=0.8972, clf=0.0519, val PR-AUC=0.0328
epoch 10: total=0.7627, recon=0.7600, clf=0.0027, val PR-AUC=0.0314
early stopping; best epoch 4
selected epoch 4 validation PR-AUC 0.032819229605837844


In [15]:
val_probs,val_targets=predict_probs(model,val_loader)
prec_curve,rec_curve,thresholds=precision_recall_curve(val_targets,val_probs)
f1s=2*prec_curve[:-1]*rec_curve[:-1]/np.maximum(prec_curve[:-1]+rec_curve[:-1],1e-12)
if not len(thresholds): raise RuntimeError("No validation thresholds")
idx=int(np.nanargmax(f1s)); THRESHOLD=float(thresholds[idx]); VAL_F1=float(f1s[idx])
print("frozen threshold",THRESHOLD,"validation F1",VAL_F1)

frozen threshold 0.0017094648210331798 validation F1 0.08333333333333333


In [16]:
test_probs,test_targets=predict_probs(model,test_loader)
test_preds=(test_probs>=THRESHOLD).astype(int)
tn,fp,fn,tp=confusion_matrix(test_targets,test_preds,labels=[0,1]).ravel()
metrics={"threshold":THRESHOLD,"test_n":int(len(test_targets)),"test_prevalence":float(test_targets.mean()),
 "precision":float(precision_score(test_targets,test_preds,zero_division=0)),
 "recall":float(recall_score(test_targets,test_preds,zero_division=0)),
 "F1":float(f1_score(test_targets,test_preds,zero_division=0)),
 "PR_AUC_average_precision":float(average_precision_score(test_targets,test_probs)),
 "ROC_AUC":float(roc_auc_score(test_targets,test_probs)) if len(np.unique(test_targets))>1 else float("nan"),
 "confusion_matrix":{"TN":int(tn),"FP":int(fp),"FN":int(fn),"TP":int(tp)},
 "base_rate_PR_AUC_reference":float(test_targets.mean()),"selected_epoch":int(best_epoch),
 "validation_PR_AUC_at_selected_epoch":float(best_val_ap)}
print(json.dumps(metrics,indent=2))
with open("test_metrics.json","w") as f: json.dump(metrics,f,indent=2)
print("Compare PR-AUC to test prevalence; do not retune using test results.")

{
  "threshold": 0.0017094648210331798,
  "test_n": 314,
  "test_prevalence": 0.054140127388535034,
  "precision": 0.06832298136645963,
  "recall": 0.6470588235294118,
  "F1": 0.12359550561797752,
  "PR_AUC_average_precision": 0.08365079272459824,
  "ROC_AUC": 0.6110120816003168,
  "confusion_matrix": {
    "TN": 147,
    "FP": 150,
    "FN": 6,
    "TP": 11
  },
  "base_rate_PR_AUC_reference": 0.054140127388535034,
  "selected_epoch": 4,
  "validation_PR_AUC_at_selected_epoch": 0.032819229605837844
}
Compare PR-AUC to test prevalence; do not retune using test results.


In [17]:
Path("artifacts").mkdir(exist_ok=True)
torch.save({"model_state_dict":model.state_dict(),"input_dim":int(X_train.shape[1]),"hidden":HP["hidden"],
 "latent":HP["latent"],"feature_columns":FEATURE_COLS,"threshold":THRESHOLD,"selected_epoch":best_epoch,"seed":SEED},
 "artifacts/wafer_guard_model.pt")
joblib.dump({"imputer":imputer,"scaler":scaler,"continuous_sensor_columns":SENSOR_COLS,
 "kept_continuous_columns":kept_cont_cols,"dropped_constant_columns":dropped_constant_cols,
 "feature_columns":FEATURE_COLS,"legacy_rule":"is_legacy=(sensor_0==0.0); set continuous sensor_0/sensor_1 to NaN for legacy rows",
 "legacy_flag_column":"is_legacy","legacy_code_columns":CODE_COLS,"threshold":THRESHOLD},
 "artifacts/preprocessing_and_threshold.joblib")
print("Artifacts:",list(Path("artifacts").iterdir()))

Artifacts: [PosixPath('artifacts/wafer_guard_model.pt'), PosixPath('artifacts/preprocessing_and_threshold.joblib')]


In [18]:
# The original notebook was run separately. Its captured outputs are recorded below;
# they must remain explicitly labelled invalid because SMOTE preceded the split.
baseline = {
    "F1": 0.9915,
    "precision": 0.9832,
    "recall": 1.0000,
    "confusion_matrix": "TN=288, FP=5, FN=0, TP=293",
    "test_prevalence": 0.5000,
    "test_n": 586,
}
audit=f"""# Wafer Guard Audit

## Findings ranked by severity

1. **SMOTE before split — critical leakage / invalid evaluation.** The original notebook preprocesses and applies SMOTE to all rows before splitting. Its reported test set has {baseline['test_n']} samples and {baseline['test_prevalence']:.2%} defect prevalence, so it contains synthetic examples and is not a natural-prevalence holdout. The impressive baseline metrics below are invalid for estimating deployment performance.
2. **Imputer/scaler fit on all rows — leakage.** The original preprocessing estimates imputation/scaling statistics before the split. In this run, the mean absolute difference between train-only and full-data imputer statistics is {statdiff['imputer_abs_diff'].mean():.4f} (median {statdiff['imputer_abs_diff'].median():.4f}); the mean absolute scaler-mean difference is {statdiff['scaler_mean_abs_diff'].mean():.4f} (median {statdiff['scaler_mean_abs_diff'].median():.4f}). These are diagnostics across 590 sensors; large sensor-specific differences are shown in Step 4.2.
3. **Random split ignores time — temporal validity.** The original notebook uses a stratified random split. The chronological split has different prevalence, and the fixed pipeline keeps the latest 20% untouched for test evaluation.
4. **Reconstruction broadcast bug — incorrect gradients.** The original loader uses [B, 1, F] while reconstruction is [B, F], so subtraction broadcasts to [B, B, F]. Step 4.4 verifies this. The corrected pipeline uses [B, F] and asserts matching shapes.
5. **PS-L legacy sentinel/codes mishandled — semantic invariant blindness.** The original pipeline treats `sensor_0 == 0.0` and legacy `sensor_1` codes 100/200/300 as continuous values. The corrected pipeline preserves `is_legacy`, one-hot encodes the codes, and masks legacy sensor_0/sensor_1 before continuous imputation/scaling.
6. **Fixed 0.5 threshold and balanced test set.** The original notebook uses a fixed 0.5 threshold after SMOTE-before-split. The corrected pipeline selects its threshold using validation data only and evaluates the frozen threshold once on the chronological test set.

## Reported original-notebook metrics — INVALID

These are the observed outputs from the uploaded, executed `bnypi (1).ipynb`; they are retained to document the failure mode, not as evidence of real-world performance.

- F1: {baseline['F1']:.4f}
- Precision: {baseline['precision']:.4f}
- Recall: {baseline['recall']:.4f}
- Confusion matrix: {baseline['confusion_matrix']}
- Test defect rate: {baseline['test_prevalence']:.2%} ({baseline['test_n']} test samples)

**Why invalid:** SMOTE was applied before splitting, so synthetic/resampled data contaminated the test set. The test prevalence was forced to 50%, versus the source dataset's natural prevalence of {y.mean():.2%}. Do not compare these scores directly with the corrected test scores or use them to justify release.

## Diagnostic comparisons

- Original-data prevalence: {y.mean():.4%} ({int(y.sum())} positives / {len(y)} rows).
- SMOTE-before-split diagnostic test prevalence: {y_test_bad.mean():.4%}; this is deliberately marked invalid.
- A train-only preprocessing + train-only SMOTE logistic-regression diagnostic on a natural-prevalence random holdout: prevalence {good_result['prevalence']:.4%}, F1 {good_result['F1']:.4f}, precision {good_result['precision']:.4f}, recall {good_result['recall']:.4f}.
- The above diagnostic scores are **not apples-to-apples** because the invalid and valid diagnostic rows use different holdout populations. They are illustrative leakage diagnostics, not a controlled estimate of the leakage effect.
- Train-only versus full-data preprocessing statistics differ; see Step 4.2 for summary and largest per-sensor differences.
- Chronological split diagnostic: random holdout prevalence {random_result['prevalence']:.4%}, F1@0.5 {random_result['F1@0.5']:.4f}, PR-AUC {random_result['PR-AUC']:.4f}; chronological last-20% prevalence {time_result['prevalence']:.4%}, F1@0.5 {time_result['F1@0.5']:.4f}, PR-AUC {time_result['PR-AUC']:.4f}. These simple logistic-regression diagnostics are not the final model and should be interpreted cautiously.

## Corrected chronological split and test metrics

- Split sizes: train {len(train_df)}, validation {len(val_df)}, test {len(test_df)}.
- Defect prevalence: train {y_train.mean():.4%}, validation {y_val.mean():.4%}, test {y_test.mean():.4%}, full dataset {y.mean():.4%}.
- Test size: {metrics['test_n']}
- Test prevalence / PR-AUC base-rate reference: {metrics['test_prevalence']:.6f}
- Frozen validation-selected threshold: {metrics['threshold']:.8f}
- Precision: {metrics['precision']:.6f}
- Recall: {metrics['recall']:.6f}
- F1: {metrics['F1']:.6f}
- PR-AUC (average precision): {metrics['PR_AUC_average_precision']:.6f}
- ROC-AUC: {metrics['ROC_AUC']:.6f}
- Confusion matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}
- Selected epoch: {metrics['selected_epoch']}
- Validation PR-AUC at selected epoch: {metrics['validation_PR_AUC_at_selected_epoch']:.6f}

**Interpretation:** the corrected test PR-AUC is only modestly above the test prevalence baseline. At the validation-selected threshold, {tp} of {tp+fn} test defects are detected, with {fp} false positives. The threshold is frozen before test evaluation; do not retune against test results. Review the operational cost of false positives and false negatives before any deployment decision.

## Required question: what are the three silent bugs?

1. **Leakage:** preprocessing statistics and SMOTE are computed before the split, contaminating evaluation.
2. **Loss gradients:** tensor broadcasting changes reconstruction loss into cross-example comparisons and corrupts gradients.
3. **PS-L handling:** sentinels and legacy categorical codes are treated as continuous values, losing their semantic meaning.

Additional issues: random rather than chronological split; fixed 0.5 threshold; accuracy is not an appropriate primary metric for rare defects.

## Residual risks

- The dataset has only {int(y.sum())} positives, and the test set has {int(y_test.sum())}; metrics are consequently noisy.
- Temporal drift may reduce future performance.
- Validation has only {int(y_val.sum())} positives, so the selected threshold and validation PR-AUC are unstable.
- The legacy subset may be too small for stable subgroup estimates.
- Before deployment, add confidence intervals, rolling temporal backtests, threshold/cost analysis, monitoring and rollback criteria.

## Go / no-go

**NO-GO.** The corrected results are weak and false positives are numerous. Do not deploy until the process owner reviews natural-prevalence performance, false-negative/false-positive costs, and prospective temporal validation. The invalid original metrics must not be used to justify release.
"""
Path("AUDIT.md").write_text(audit,encoding="utf-8")
print("Wrote AUDIT.md with observed invalid baseline and corrected metrics.")


Wrote AUDIT.md with observed invalid baseline and corrected metrics.


In [19]:
print({
    "approved_csv": DATA_PATH.exists(),
    "metrics_json": Path("test_metrics.json").exists(),
    "audit": Path("AUDIT.md").exists(),
    "model_artifact": Path("artifacts/wafer_guard_model.pt").exists(),
    "preprocessing_artifact": Path("artifacts/preprocessing_and_threshold.joblib").exists(),
})
print("Submission checklist: name the notebook wafer_guard_fixed.ipynb in your repo; restart the kernel and run all cells; confirm AUDIT.md and test_metrics.json match this run; do not upload evaluator-only CSV.")


{'approved_csv': True, 'metrics_json': True, 'audit': True, 'model_artifact': True, 'preprocessing_artifact': True}
Submission checklist: name the notebook wafer_guard_fixed.ipynb in your repo; restart the kernel and run all cells; confirm AUDIT.md and test_metrics.json match this run; do not upload evaluator-only CSV.
